# Case Study 1: Credit Scoring

Audits a RandomForest credit-default model for fairness across a protected `region` attribute, explainability, and robustness. **Synthetic data**: no real applicant data is used anywhere in this repository. The income gap between regions is deliberately baked into the generator to simulate a historical-bias scenario a real audit would need to catch.

In [1]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

from responsible_ai_finance import audit

pd.set_option("display.precision", 3)
np.random.seed(42)


In [2]:
rng = np.random.default_rng(7)
n = 1500

income = rng.normal(34000, 11000, n).clip(6000, None)
debt_ratio = rng.uniform(0, 1, n)
age = rng.integers(18, 75, n)
utilization = rng.uniform(0, 1, n)
region = rng.choice(["Region_A", "Region_B"], size=n, p=[0.55, 0.45])

# Simulated historical bias: Region_B applicants show a systematically lower
# recorded income for otherwise similar risk profiles (a common real-world
# proxy-discrimination pattern this audit should surface).
income_adj = income - np.where(region == "Region_B", 3500, 0)

logit = -0.2 - income_adj / 15000 + 2.3 * debt_ratio + 1.4 * utilization - 0.01 * age
prob_default = 1 / (1 + np.exp(-logit))
y = (rng.uniform(0, 1, n) < prob_default).astype(int)

X = pd.DataFrame({"income": income, "debt_ratio": debt_ratio, "age": age, "utilization": utilization})
X_train, X_test, y_train, y_test, region_train, region_test = train_test_split(
    X, y, region, test_size=0.3, random_state=42, stratify=y
)

model = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)
model.fit(X_train, y_train)
print(f"Train accuracy: {model.score(X_train, y_train):.3f}")
print(f"Test accuracy:  {model.score(X_test, y_test):.3f}")
print(f"Default rate in test set: {y_test.mean():.3f}")


Train accuracy: 0.834
Test accuracy:  0.733
Default rate in test set: 0.316


In [3]:
report = audit(
    model,
    X_test.reset_index(drop=True),
    pd.Series(y_test).reset_index(drop=True),
    pd.Series(region_test).reset_index(drop=True),
    model_name="credit-scoring-rf",
)
print(report.to_markdown())


# Responsible AI Audit — credit-scoring-rf
_Generated 2026-09-13T11:37:37.979776+00:00_

## Governance Flags
- ⚠️ Zeroing 'income' flips 33.5% of predictions — check upstream data-quality guarantees for this feature.

## Fairness

- Demographic parity difference: **0.0267**
- Disparate impact ratio: **0.8667** (80% rule: PASS)
- Equalized odds — TPR difference: **0.0953**
- Equalized odds — FPR difference: **0.0130**

| Group | n | Selection rate | TPR | FPR |
|---|---|---|---|---|
| Region_A | 225 | 0.200 | 0.424 | 0.107 |
| Region_B | 225 | 0.173 | 0.329 | 0.094 |

## Explainability

- Top features by mean |SHAP value|: debt_ratio, income, utilization, age
- SHAP local-fidelity MAE: **0.0000**

## Robustness

- Prediction flip rate under 5% Gaussian noise: **2.0%**
- Most fragile feature to dropout: **income** (33.5% flip rate)


## Key Takeaways

- This notebook is executed end to end on every run — the numbers above are real outputs of this code, not hand-typed placeholders.
- The governance flags section is the first thing a reviewer should read; the detailed tables below it exist to let them verify the flag.
- Whether the disparate-impact flag fires depends on the specific random split and model fit — re-run the notebook to see this.